In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz
from ipywidgets import HTML
from IPython.display import display

# ============================================================
# POLE-ZERO PLACEMENT
# SECOND-ORDER BAND-PASS AND BAND-STOP EXAMPLES
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.pz-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.pz-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.pz-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.pz-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.pz-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
    margin-bottom:5px;
}

.pz-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.pz-col{
    flex:1;
    min-width:0;
}

.pz-note{
    background:#fff9e8;
    border:1px solid #d9c477;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="pz-root">

<div class="pz-header">
Second-Order Band-Pass and Band-Stop Filters by Pole-Zero Placement
</div>

<div class="pz-doc">

This notebook reproduces the second pole-zero placement example using

<b>f₀ = 1200 Hz</b>,
<b>BW = 220 Hz</b>, and
<b>fₛ = 8500 Hz</b>.

The conjugate poles are placed at

<div style="text-align:center;font-size:15.5px;margin:6px 0;">
<b>
p₁,₂ = r e<sup>±jθ</sup>,
&nbsp;&nbsp;
r ≈ 1 - πBW/fₛ,
&nbsp;&nbsp;
θ = 2πf₀/fₛ.
</b>
</div>

For the <b>band-pass filter</b>, zeros are placed at <b>z = +1</b> and <b>z = -1</b>,
forcing zero response at DC and at the Nyquist frequency.

For the <b>band-stop filter</b>, the zeros are placed on the unit circle at the
same angles as the poles,

<div style="text-align:center;font-size:15.5px;margin:6px 0;">
<b>z₁,₂ = e<sup>±jθ</sup>,</b>
</div>

forcing a notch at the center frequency.

The two filters use the same pole pair; only the zero locations and normalization
constants are different.

</div>

</div>
"""))

# ============================================================
# SPECIFICATIONS
# ============================================================

f0 = 1200.0
BW = 220.0
fs = 8500.0

# ============================================================
# POLE RADIUS AND ANGLE
# ============================================================

r = 1.0-np.pi*BW/fs

theta = 2.0*np.pi*f0/fs

theta_deg = np.degrees(theta)

theta_norm = theta/np.pi

# ============================================================
# CONJUGATE POLES
# ============================================================

p1 = r*np.exp(1j*theta)

p2 = r*np.exp(-1j*theta)

poles = np.array([p1,p2])

# ============================================================
# COMMON DENOMINATOR
#
# 1 - 2 r cos(theta) z^-1 + r^2 z^-2
# ============================================================

a = np.array([1.0,-2.0*r*np.cos(theta),r**2])

# ============================================================
# BAND-PASS FILTER
# ============================================================

zeros_bp = np.array([1.0+0j,-1.0+0j])

H0_bp = (1.0-r)*np.sqrt(1.0-2.0*r*np.cos(2.0*theta)+r**2)/(2.0*np.abs(np.sin(theta)))

b_bp = H0_bp*np.array([1.0,0.0,-1.0])

# ============================================================
# BAND-STOP FILTER
# ============================================================

zeros_bs = np.array([np.exp(1j*theta),np.exp(-1j*theta)])

H0_bs = (1.0-2.0*r*np.cos(theta)+r**2)/(2.0*(1.0-np.cos(theta)))

b_bs = H0_bs*np.array([1.0,-2.0*np.cos(theta),1.0])

# ============================================================
# FREQUENCY RESPONSES
# ============================================================

omega_bp,H_bp = freqz(b_bp,a,worN=32768)

omega_bs,H_bs = freqz(b_bs,a,worN=32768)

fn_bp = omega_bp/np.pi

fn_bs = omega_bs/np.pi

mag_bp = np.abs(H_bp)

mag_bs = np.abs(H_bs)

# ============================================================
# RESPONSE AT CENTER FREQUENCY
# ============================================================

def evaluate_Hz(b,a,omega):

    q = np.exp(-1j*omega)

    numerator = np.sum(b*q**np.arange(len(b)))

    denominator = np.sum(a*q**np.arange(len(a)))

    return numerator/denominator

H_bp_center = evaluate_Hz(b_bp,a,theta)

H_bs_center = evaluate_Hz(b_bs,a,theta)

mag_bp_center = np.abs(H_bp_center)

mag_bs_center = np.abs(H_bs_center)

# ============================================================
# NORMALIZATION CHECK FOR BAND-STOP AT DC
# ============================================================

H_bs_dc = evaluate_Hz(b_bs,a,0.0)

mag_bs_dc = np.abs(H_bs_dc)

# ============================================================
# NUMERICAL RESULTS
# ============================================================

display(HTML(f"""
<div class="pz-root">

<div class="pz-box">

<div class="pz-title">Numerical results</div>

<div class="pz-cols">

<div class="pz-col">

<b>Operating parameters</b><br>

f₀ = <b>{f0:.0f} Hz</b><br>
BW = <b>{BW:.0f} Hz</b><br>
fₛ = <b>{fs:.0f} Hz</b><br><br>

r = <b>{r:.9f}</b><br>
θ = <b>{theta:.9f} rad</b><br>
θ = <b>{theta_deg:.6f}°</b><br>
θ/π = <b>{theta_norm:.9f}</b>

</div>

<div class="pz-col">

<b>Common conjugate poles</b><br>

p₁ = <b>{np.real(p1):.9f} {np.imag(p1):+.9f}j</b><br>
p₂ = <b>{np.real(p2):.9f} {np.imag(p2):+.9f}j</b><br><br>

Denominator:<br>

<b>
1 {a[1]:+.9f}z<sup>-1</sup>
{a[2]:+.9f}z<sup>-2</sup>
</b>

</div>

<div class="pz-col">

<b>Normalization constants</b><br>

Band-pass:<br>
H₀ = <b>{H0_bp:.9f}</b><br><br>

Band-stop:<br>
H₀ = <b>{H0_bs:.9f}</b><br><br>

Center frequency:<br>
ω₀ = <b>{theta_norm:.6f}π</b>

</div>

</div>

</div>

</div>
"""))

# ============================================================
# TRANSFER FUNCTIONS
# ============================================================

display(HTML(f"""
<div class="pz-root">

<div class="pz-box pz-note">

<div class="pz-title">Transfer functions and numerical checks</div>

<div class="pz-cols">

<div class="pz-col">

<b>Band-pass filter</b>

<div style="font-size:14.5px;margin:6px 0;line-height:1.65;">

H(z) =
<b>
({b_bp[0]:.6f}
{b_bp[1]:+.6f}z<sup>-1</sup>
{b_bp[2]:+.6f}z<sup>-2</sup>) /
</b>

<br>

<div style="padding-left:48px;">
<b>
(1
{a[1]:+.6f}z<sup>-1</sup>
{a[2]:+.6f}z<sup>-2</sup>)
</b>
</div>

</div>

At ω = θ:<br>
<b>|H(e<sup>jθ</sup>)| = {mag_bp_center:.9f}</b>

</div>

<div class="pz-col">

<b>Band-stop filter</b>

<div style="font-size:14.5px;margin:6px 0;line-height:1.65;">

H(z) =
<b>
({b_bs[0]:.6f}
{b_bs[1]:+.6f}z<sup>-1</sup>
{b_bs[2]:+.6f}z<sup>-2</sup>) /
</b>

<br>

<div style="padding-left:48px;">
<b>
(1
{a[1]:+.6f}z<sup>-1</sup>
{a[2]:+.6f}z<sup>-2</sup>)
</b>
</div>

</div>

At ω = θ:<br>
<b>|H(e<sup>jθ</sup>)| = {mag_bs_center:.3e}</b>

&nbsp;&nbsp;&nbsp;

At ω = 0:<br>
<b>|H(1)| = {mag_bs_dc:.9f}</b>

</div>

</div>

</div>

</div>
"""))

# ============================================================
# FIGURE — 2 x 2 GRID
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.4))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

theta_circle = np.linspace(0,2*np.pi,1000)

unit_x = np.cos(theta_circle)

unit_y = np.sin(theta_circle)

# ============================================================
# 1. BAND-PASS POLE-ZERO PLOT
# ============================================================

ax1.axhline(0,color='black',linewidth=0.8)

ax1.axvline(0,color='black',linewidth=0.8)

ax1.plot(unit_x,unit_y,'--',linewidth=1.0,label='Unit circle')

ax1.plot(np.real(poles),np.imag(poles),'rx',markersize=8,markeredgewidth=1.8,label='Poles')

ax1.plot(np.real(zeros_bp),np.imag(zeros_bp),'bo',markersize=7,markerfacecolor='none',markeredgewidth=1.6,label='Zeros')

ax1.set_xlim(-1.2,1.2)

ax1.set_ylim(-1.2,1.2)

ax1.set_aspect('equal',adjustable='box')

ax1.set_title('Band-Pass Pole-Zero Plot')

ax1.set_xlabel(r'$\Re\{z\}$')

ax1.set_ylabel(r'$\Im\{z\}$')

ax1.grid(True,linestyle=':',alpha=0.25)

ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=3,frameon=False)

# ============================================================
# 2. BAND-PASS MAGNITUDE RESPONSE
# ============================================================

ax2.plot(fn_bp,mag_bp,color='red',linewidth=1.5,label='Band-pass response')

ax2.axvline(theta_norm,linestyle='--',linewidth=1.0,label=r'$\omega_0$')

ax2.plot([theta_norm],[mag_bp_center],'o',markersize=5)

ax2.set_xlim(0,1)

ax2.set_ylim(0,1.08)

ax2.set_title('Band-Pass Magnitude Response')

ax2.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax2.set_ylabel(r'$|H(e^{j\omega})|$')

ax2.grid(True,linestyle=':',alpha=0.25)

ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 3. BAND-STOP POLE-ZERO PLOT
# ============================================================

ax3.axhline(0,color='black',linewidth=0.8)

ax3.axvline(0,color='black',linewidth=0.8)

ax3.plot(unit_x,unit_y,'--',linewidth=1.0,label='Unit circle')

ax3.plot(np.real(poles),np.imag(poles),'rx',markersize=8,markeredgewidth=1.8,label='Poles')

ax3.plot(np.real(zeros_bs),np.imag(zeros_bs),'bo',markersize=7,markerfacecolor='none',markeredgewidth=1.6,label='Zeros')

ax3.set_xlim(-1.2,1.2)

ax3.set_ylim(-1.2,1.2)

ax3.set_aspect('equal',adjustable='box')

ax3.set_title('Band-Stop Pole-Zero Plot')

ax3.set_xlabel(r'$\Re\{z\}$')

ax3.set_ylabel(r'$\Im\{z\}$')

ax3.grid(True,linestyle=':',alpha=0.25)

ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=3,frameon=False)

# ============================================================
# 4. BAND-STOP MAGNITUDE RESPONSE
# ============================================================

ax4.plot(fn_bs,mag_bs,color='red',linewidth=1.5,label='Band-stop response')

ax4.axvline(theta_norm,linestyle='--',linewidth=1.0,label=r'$\omega_0$')

ax4.plot([theta_norm],[mag_bs_center],'o',markersize=5)

ax4.set_xlim(0,1)

ax4.set_ylim(0,1.08)

ax4.set_title('Band-Stop Magnitude Response')

ax4.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax4.set_ylabel(r'$|H(e^{j\omega})|$')

ax4.grid(True,linestyle=':',alpha=0.25)

ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# LAYOUT
# ============================================================

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.28,hspace=0.56)

# ============================================================
# DISPLAY
# ============================================================

display(fig.canvas)